# Homework: Tracks (14/10/2026)

In [23]:

import sqlite3

# Conecta con la base de datos SQLite llamada 'trackdb.sqlite'.
# Si el archivo no existe, Python lo crea automáticamente.
conn = sqlite3.connect('trackdb.sqlite')

# Crea un cursor para ejecutar instrucciones SQL sobre la base de datos.
cur = conn.cursor()


# ============================================================
# 1. ELIMINAR LAS TABLAS SI YA EXISTEN
# ============================================================

# Elimina la tabla Artist si existe.
# Esto permite comenzar desde cero al ejecutar el programa.
cur.execute('''
DROP TABLE IF EXISTS Artist''')

# Elimina la tabla Genre si existe.
cur.execute('''
DROP TABLE IF EXISTS Genre''')

# Elimina la tabla Album si existe.
cur.execute('''
DROP TABLE IF EXISTS Album''')

# Elimina la tabla Track si existe.
cur.execute('''
DROP TABLE IF EXISTS Track''')


# ============================================================
# 2. CREAR LAS TABLAS DE LA BASE DE DATOS
# ============================================================

# Crea la tabla Artist para almacenar los artistas musicales.
cur.execute('''
CREATE TABLE Artist (
    id  INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT UNIQUE,
    name    TEXT UNIQUE
)''')

# Crea la tabla Genre para almacenar los géneros musicales.
cur.execute('''
CREATE TABLE Genre (
    id  INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT UNIQUE,
    name    TEXT UNIQUE
)''')

# Crea la tabla Album para almacenar los álbumes.
# artist_id identifica al artista asociado con cada álbum.
cur.execute('''
CREATE TABLE Album (
    id  INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT UNIQUE,
    artist_id  INTEGER,
    title   TEXT UNIQUE
)''')

# Crea la tabla Track para almacenar las canciones.
# album_id identifica el álbum de cada canción.
# genre_id identifica el género musical de cada canción.
# len almacena la duración, rating la calificación y count el número de reproducciones.
cur.execute('''
CREATE TABLE Track (
    id  INTEGER NOT NULL PRIMARY KEY 
        AUTOINCREMENT UNIQUE,
    title TEXT  UNIQUE,
    album_id  INTEGER,
    genre_id  INTEGER,
    len INTEGER, 
    rating INTEGER, 
    count INTEGER
)''')


# ============================================================
# 3. SOLICITAR EL NOMBRE DEL ARCHIVO
# ============================================================

# Pide al usuario el nombre del archivo que contiene los datos.
fname = input('Enter file name: ')

# Si el usuario no escribe ningún nombre, utiliza 'tracks.csv'.
if (len(fname) < 1):
    fname = 'tracks.csv'

# Abre el archivo para leer su contenido.
fh = open(fname)


# ============================================================
# 4. LEER Y PROCESAR CADA LÍNEA DEL ARCHIVO
# ============================================================

# Ejemplo de una línea del archivo:
# Another One Bites The Dust,Queen,Greatest Hits,55,100,217103,Rock
# 0                          1     2             3  4   5      6

# Recorre el archivo línea por línea.
for line in fh:

    # Elimina espacios y saltos de línea al principio y al final.
    line = line.strip()

    # Divide la línea en una lista usando la coma como separador.
    pieces = line.split(',')

    # Si la línea tiene menos de 7 elementos, la ignora
    # y pasa a la siguiente iteración del ciclo.
    if len(pieces) < 7:
        continue

    # Guarda cada elemento de la lista en una variable.
    track = pieces[0]   # Nombre de la canción.
    artist = pieces[1]  # Nombre del artista.
    album = pieces[2]   # Nombre del álbum.
    count = pieces[3]   # Número de reproducciones.
    rating = pieces[4]  # Calificación de la canción.
    length = pieces[5]  # Duración de la canción.
    genre = pieces[6]   # Género musical.

    # Esta instrucción, si se descomenta, muestra los datos leídos.
    # print(track, artist, album, count, rating, length, genre)


    # ========================================================
    # 5. INSERTAR EL ARTISTA Y OBTENER SU ID
    # ========================================================

    # Inserta el artista si todavía no existe.
    # INSERT OR IGNORE evita insertar un duplicado si el nombre
    # ya existe, porque la columna name es UNIQUE.
    cur.execute('''INSERT OR IGNORE INTO Artist (name)
        VALUES ( ? )''', (artist, ))

    # Busca el ID del artista usando su nombre.
    # El signo ? se reemplaza de forma segura por el valor artist.
    cur.execute('SELECT id FROM Artist WHERE name = ? ', (artist, ))

    # Recupera la fila del resultado y extrae su primer valor: id.
    artist_id = cur.fetchone()[0]


    # ========================================================
    # 6. INSERTAR EL GÉNERO Y OBTENER SU ID
    # ========================================================

    # Inserta el género si todavía no existe.
    cur.execute('''INSERT OR IGNORE INTO Genre (name)
        VALUES ( ? )''', (genre, ))

    # Busca el ID correspondiente al nombre del género.
    cur.execute('SELECT id FROM Genre WHERE name = ? ', (genre, ))

    # Obtiene el ID del género encontrado.
    genre_id = cur.fetchone()[0]


    # ========================================================
    # 7. INSERTAR EL ÁLBUM Y OBTENER SU ID
    # ========================================================

    # Inserta el álbum junto con el ID de su artista,
    # siempre que el título del álbum no exista ya.
    cur.execute('''INSERT OR IGNORE INTO Album (artist_id, title)
        VALUES ( ?, ?)''', (artist_id, album))

    # Busca el ID del álbum mediante su título.
    cur.execute('SELECT id FROM Album WHERE title = ? ', (album, ))

    # Obtiene el ID del álbum encontrado.
    album_id = cur.fetchone()[0]


    # ========================================================
    # 8. INSERTAR LA CANCIÓN EN LA TABLA TRACK
    # ========================================================

    # Inserta la canción con sus datos y los ID asociados.
    # INSERT OR REPLACE reemplaza una fila existente si ocurre
    # un conflicto con una restricción UNIQUE, por ejemplo,
    # si ya existe una canción con el mismo título.
    cur.execute('''INSERT OR REPLACE INTO Track
        (title, album_id, genre_id, len, rating, count)
        VALUES ( ?, ?, ?, ?, ?, ?)''',
        (track, album_id, genre_id, length, rating, count))


# ============================================================
# 9. GUARDAR LOS CAMBIOS
# ============================================================

# Confirma los cambios realizados en la base de datos.
# Sin esta confirmación, los cambios podrían no quedar guardados.
conn.commit()

# Cierra el archivo después de terminar de leerlo.
fh.close()

# Cierra la conexión con la base de datos.
conn.close()

Enter file name: 


In [25]:
import sqlite3

# ============================================================
# 1. CONECTARSE A LA BASE DE DATOS
# ============================================================

# Abre una conexión con la base de datos llamada 'trackdb.sqlite'.
# Si el archivo no existe, SQLite lo crea automáticamente.
conn = sqlite3.connect('trackdb.sqlite')

# Crea un cursor para ejecutar consultas SQL en la base de datos.
cur = conn.cursor()


# ============================================================
# 2. EJECUTAR LA CONSULTA SQL
# ============================================================

# Selecciona cuatro columnas:
# - Track.title: nombre de la canción.
# - Artist.name: nombre del artista.
# - Album.title: nombre del álbum.
# - Genre.name: nombre del género musical.
#
# FROM Track indica que la consulta comienza en la tabla Track.
#
# JOIN Genre relaciona cada canción con su género musical,
# utilizando Track.genre_id = Genre.id.
#
# JOIN Album relaciona cada canción con su álbum,
# utilizando Track.album_id = Album.id.
#
# JOIN Artist relaciona cada álbum con su artista,
# utilizando Album.artist_id = Artist.id.
#
# ORDER BY Artist.name ordena los resultados alfabéticamente
# según el nombre del artista.
#
# LIMIT 5 limita el resultado a un máximo de tres filas.

cur.execute('''
    SELECT Track.title, Artist.name, Album.title, Genre.name
    FROM Track
    JOIN Genre ON Track.genre_id = Genre.id
    JOIN Album ON Track.album_id = Album.id
    JOIN Artist ON Album.artist_id = Artist.id
    ORDER BY Artist.name
    LIMIT 3
''')


# ============================================================
# 3. RECUPERAR LOS RESULTADOS
# ============================================================

# Obtiene todas las filas devueltas por la consulta SQL.
# Como utilizamos LIMIT 3, se recuperan como máximo tres filas.
rows = cur.fetchall()


# ============================================================
# 4. MOSTRAR LOS RESULTADOS
# ============================================================

# Recorre cada fila de los resultados.
for row in rows:

    # Imprime los datos de cada canción.
    # Cada fila es una tupla con cuatro valores:
    # título de la canción, artista, álbum y género.
    print(row)


# ============================================================
# 5. CERRAR LA CONEXIÓN
# ============================================================

# Cierra la conexión con la base de datos.
conn.close()

('For Those About To Rock (We Salute You)', 'AC/DC', 'Who Made Who', 'Rock')
('Hells Bells', 'AC/DC', 'Who Made Who', 'Rock')
('Shake Your Foundations', 'AC/DC', 'Who Made Who', 'Rock')
